# Unsloth Fine-tuning DeepSeek R1 Distilled Llama 8B

In this notebook, it will demonstrate how to finetune `DeepSeek-R1-Distill-Llama-8B` with Unsloth, using a medical dataset.

## Why do we need LLM fine-tuning?

Fine-tuning tailors the model to have a better performance for specific tasks, making it more effective and versatile in real-world applications. This process is essential for tailoring an existing model to a particular task or domain.

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install bitsandbytes unsloth_zoo


## Choose a Base Model

1. Choose a model that aligns with your usecase
2. Assess your storage, compute capacity and dataset
3. Select a Model and Parameters
4. Choose Between Base and Instruct Models

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.2.12: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA L4. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Inference before fine-tuning

In [ ]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>{}"""

In [ ]:
question = "一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？"


FastLanguageModel.for_inference(model)
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)#这一步是没有微调前进行一个推理
print(response[0].split("### Response:")[1])


<think>
好的，我现在需要处理一个关于急性阑尾炎的病人。病人已经发病5天，腹痛有所减轻，但仍然发热。在体检时，医生发现了右下腹有压痛的包块。那么，如何处理这个情况呢？

首先，急性阑尾炎通常是由于细菌感染引起的，常见于儿童和青年人，发病期一般为3-5天。病人已经有5天的病史，说明炎症已经持续了一段时间。腹痛虽然有所减轻，但仍然发热，这可能意味着炎症正在缓解，但仍未完全消退。

接下来，体检发现右下腹有压痛的包块，这是一个重要的发现。包块的存在可能意味着腹腔内有一个明显的结构性改变。压痛通常是指在触摸或压迫特定区域时，患者会感到疼痛或不适。包块的存在可能提示阑尾炎的一些并发症，比如阑尾的炎症延伸（如腺体炎或胰腺炎），或者是其他相邻脏器的炎症。

考虑到这些因素，首先应该进行的检查包括：

1. **腹部超声检查**：为了更清晰地了解包块的性质，如是否是液体积或固体积，以及周围组织的反应。超声可以帮助确定包块的具体位置和形态，是否有积液或是腺体的增大。

2. **血液检查**：包括白细胞计数、C反应蛋白和血培养，评估感染的严重程度和存在的细菌感染情况。

3. **粪便细菌培养**：如果有腹泻或便秘，或者考虑是否存在感染性疾病。

4. **胸部X光检查**：如果有包块可能在肺部，但在这种情况下，更可能是腹腔内的问题。

5. **腹腔镜或腹腔穿刺检查**：如果包块的性质不明确，或者需要获得病理学上的证据，可能需要进行腹腔镜或穿刺检查。

在处理急性阑尾炎时，通常的治疗包括抗生素治疗，根据敏感性来选择药物。同时，饮食建议、补液以及可能的解除病痛措施如热敷或药物缓解。

然而，包块的存在可能意味着需要进一步的评估和治疗。例如，如果包块是脓肿，可能需要引流治疗。或者，如果包块是腺体炎，可能需要特定的药物治疗。此外，如果包块周围有脓液，可能需要抗生素治疗以防止感染扩散。

另外，考虑到病人已经有5天的发热和腹痛，可能需要考虑是否存在并发症，如感染性休克、多器官功能障碍等，这些都需要及时的处理。

总结一下，处理步骤应该是：

1. 进行腹部超声检查以明确包块的性质。
2. 完成血液和细菌培养，评估感染情况。
3. 根据检查结果决定是否进行腹腔镜或穿刺检查。
4. 根据包块性质和病情，决定是否需要引流或其他治疗措施。
5. 综合治疗包括抗生素、支持治疗和可能的解除病痛

## Prepare Dataset

A medical dataset [https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/) will be used to train the selected model.

In [ ]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

### Important Notice

It's crucial to add the EOS (End of Sequence) token at the end of each training dataset entry, otherwise you may encounter infinite generations.

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN


def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [ ]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'zh', split = "train[:80%]", trust_remote_code=True)
print(dataset.column_names)

README.md:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

medical_o1_sft_Chinese.json:   0%|          | 0.00/64.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24772 [00:00<?, ? examples/s]

['Question', 'Complex_CoT', 'Response']


For `Ollama` and `llama.cpp` to function like a custom `ChatGPT` Chatbot, we must only have 2 columns - an `instruction` and an `output` column. We need to transform the dataset into proper structure.

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset["text"][0]

Map:   0%|          | 0/19818 [00:00<?, ? examples/s]

'Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request.\nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\nPlease answer the following medical question.\n\n### Question:\n根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？\n\n### Response:\n<think>\n这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本

## Train the model
Now let's use Huggingface TRL's `SFTTrainer`.

In [ ]:
FastLanguageModel.for_training(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((409

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 600,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc，none是不用任何报告插件
    ),
)

Applying chat template to train dataset (num_proc=2):   0%|          | 0/19818 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/19818 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/19818 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 19,818 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 600
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,2.112200
2,2.189900
3,2.259200
4,2.032400
5,2.234900
6,2.186700
7,1.889700
8,1.920300
9,1.654400
10,1.753500


Step,Training Loss
1,2.112200
2,2.189900
3,2.259200
4,2.032400
5,2.234900
6,2.186700
7,1.889700
8,1.920300
9,1.654400
10,1.753500


## Inference after fine-tuning

Let's inference with same question again and see the difference.

In [ ]:
print(question)

In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
患者已经发病5天了，腹痛稍微减轻，但仍然发烧，这让我觉得情况可能不简单。还有右下腹有压痛的包块，这听起来像是有地方出了问题。考虑到他之前是急性阑尾炎，可能是阑尾炎复发或者有其他并发症。

我想，腹痛减轻可能是因为炎症有所缓解，但发烧依旧，说明体内可能还有炎症存在。压痛的包块可能是阑尾炎复发的信号，或者可能是其他地方，比如腹腔里有其他东西，比如脓肿或者肿瘤。

嗯，为了保险起见，我们应该考虑一些进一步的检查。腹部超声波检查是个不错的选择，因为它可以帮助我们看看包块的性质，比如是不是脓肿或者有没有液体。还有，如果超声波没给我们足够的信息，我们可以考虑做个腹腔穿刺来看看包块里到底是什么。

在处理上，我觉得要确保患者的感染控制住了。抗生素的使用很重要，因为这可以帮助控制感染的扩散。至于包块，虽然我们需要了解它的性质，但即使我们知道了，治疗可能会依赖于更多的信息。

所以，综合考虑下来，我觉得我们应该先给患者用抗生素，控制感染，然后安排腹部超声波检查，看看包块到底是什么，再根据结果来决定后续的治疗步骤。这应该能帮助我们更好地处理患者的病情。
</think>
在处理这种情况时，我们需要采取一些积极的措施来确保患者的感染得到控制并找出可能的病因。

首先，考虑到患者已经发病5天，腹痛虽然有所缓解，但仍然发热，并且右下腹有压痛的包块，这表明可能存在感染的复发或其他并发症。我们应该立即给患者使用抗生素，以帮助控制感染的扩散。这可以降低感染的风险，并减轻症状。

其次，为了了解包块的性质，我们可以建议进行腹部超声波检查。超声波可以帮助我们确定包块是否为脓肿、液体或其他结构，并提供关于包块位置、大小和边界的更多信息。如果超声波检查结果不够明确，我们可以考虑进行腹腔穿刺，获取包块内的液体进行化验，以便更准确地诊断。

在获得超声波检查和可能的腹腔穿刺结果后，我们可以根据具体的诊断结果来制定进一步的治疗计划。无论是阑尾炎复发还是其他可能的病因，我们都需要根据专业的医疗意见来决定最适合的治疗方案。

总之，首先要控制感染，确保患者的感染得到有效控制；其次是通过影像学检查和必要的进一步检查来明确病因和病情；最后根据诊断结果来制定治疗方案。这样才能为患者提供更全面的医疗服务。<｜end▁of▁sentence｜>


In [ ]:
#为了加快测试速度，所以在最后50个样本做测试
test_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'zh', split = "train[-50:]", trust_remote_code=True)
test_questions = test_dataset["Question"]
test_references = test_dataset["Response"]  # 参考答案
test_questions[0]

'患者女性,26岁。白带呈黄色脓性,伴尿急、尿痛和排尿困难。妇科检查:外阴、阴道及尿道口红肿充血,从阴道前壁压迫尿道及尿道旁腺有脓液外溢。患者最可能患了\nA. 淋病\nB. 尖锐湿疣\nC. 非特异性外阴炎\nD. 前庭大腺炎\nE. 急性宫颈炎'

In [ ]:

# 生成模型的预测结果
model_predictions = []
for question in test_questions:
    inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1200,
        use_cache=True,
    )
    response = tokenizer.batch_decode(outputs)[0].split("### Response:")[1]
    model_predictions.append(response)


KeyboardInterrupt: 

In [ ]:
model_predictions[1]

'\n<think>\n嗯，15岁的男孩，腹泻已经持续了5年，每天2次，感觉有点久啊。再加上他有里急后重的感觉，这让我想到可能是肠道有问题。还有，有时候大便会出血，这可不太妙，虽然没有发热，但这出血可能还是要小心的。\n\n粪细菌培养结果是阴性，这意味着细菌感染可能性不大，可能是非典型的原因。肠镜检查显示乙状结肠血管纹理不清，粘膜呈颗粒状，轻触就出血，这些都很典型啊。\n\n结合这些症状和检查结果，肠道可能有炎症或损伤，导致血管不稳定和粘膜脱落。想想看，非典型的肠道疾病，比如肠道粘膜病，会不会就是这个呢？\n\n再想想，肠道粘膜病（IBD）常见的表现是这种出血性腹泻，尤其是乙状结肠的改变，和这个小伙子症状很像。虽然没有发热，但出血和粪便的异常确实和IBD的表现很吻合。\n\n所以，综合来看，这个小伙子最可能的诊断就是非典型的肠道粘膜病。嗯，这样的分析应该是合理的，症状和检查都指向这个方向。\n</think>\n根据患者的症状和检查结果，最可能的诊断是非典型的肠道粘膜病（IBD）。患者的腹泻持续时间长、伴有里急后重和出血，虽然没有发热，但粪便异常和肠镜检查结果（乙状结肠血管纹理不清，粘膜呈颗粒状且轻触易出血）都指向肠道粘膜病的特征。这种疾病常表现为出血性腹泻，并且通常伴有粘膜脱落和血管不稳定。因此，综合这些信息，该患者的诊断为非典型的肠道粘膜病是合理的。<｜end▁of▁sentence｜>'

In [ ]:
len(model_predictions)

34

In [ ]:
test_references[1]

'根据您提供的症状和检查结果，这位15岁男性患者最可能的诊断是溃疡性结肠炎。患者长期腹泻、间歇性便血、里急后重感，加上肠镜检查发现乙状结肠的血管纹理不清晰、粘膜呈颗粒状且轻触易出血，这些都与溃疡性结肠炎的临床表现相符。溃疡性结肠炎是一种慢性炎症性肠病，常表现为持续或反复发作的腹泻，伴血便和腹痛。建议患者在专业医生的指导下接受进一步的诊断确认和治疗。'

In [ ]:
!pip install nltk rouge

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
#因为推理太慢了，所以推理了34个后打断了，只对已经推理的34个样本计算了bleu指标
bleu_scores = []
for pred, ref in zip(model_predictions[0:34], test_references[0:34]):
    # 将参考答案和预测结果分词
    ref_tokens = [ref.split()]
    pred_tokens = pred.split()
    # 计算 BLEU 分数
    bleu_score = sentence_bleu(ref_tokens, pred_tokens,weights=(1,0,0,0))
    bleu_scores.append(bleu_score)

# 计算平均 BLEU 分数
average_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU Score: {average_bleu}")


Average BLEU Score: 0.010689043426752545


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

AssertionError: 

下面的部分可以不做，不用上传到抱抱脸上

## Upload Model to HuggingFace

Now, let's save our finetuned model and upload it to HuggingFace.

### Save the fine-tuned model to GGUF format

Choose the llama.cpp's GGUF format we prefer by setting the corresponding `if` to `True`.

In [ ]:
from google.colab import userdata

HUGGINGFACE_TOKEN = userdata.get('HUGGINGFACE_TOKEN')

In [ ]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("model", tokenizer,)

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model_f16", tokenizer, quantization_method = "f16")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 6.0G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 34.71 out of 52.96 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 91%|█████████ | 29/32 [00:01<00:00, 18.35it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [00:02<00:00, 12.46it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be /content/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Conversion completed! Output location: /content/model/unsloth.Q8_0.gguf


### Push the model to HuggingFace

Create a model type repository for your model if you haven't done so.

In [ ]:
from huggingface_hub import create_repo
create_repo("wyang14/medical-model", token=HUGGINGFACE_TOKEN, exist_ok=True)

RepoUrl('https://huggingface.co/wyang14/medical-model', endpoint='https://huggingface.co', repo_type='model', repo_id='wyang14/medical-model')

In [ ]:
model.push_to_hub_gguf("wyang14/medical-model", tokenizer, token = HUGGINGFACE_TOKEN)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 33.04 out of 52.96 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [00:02<00:00, 13.40it/s]


Unsloth: Saving tokenizer... Done.
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at wyang14/medical-model into q8_0 GGUF format.
The output location will be /content/wyang14/medical-model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: medical-model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-0

  0%|          | 0/1 [00:00<?, ?it/s]

unsloth.Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Saved GGUF to https://huggingface.co/wyang14/medical-model


### Ollama run HuggingFace model

```bash
ollama run hf.co/{username}/{repository}:{quantization}
```

### Ollama inference

```bash
curl http://localhost:11434/api/chat -d '{ \
  "model": "", \
  "messages": [ \
    { "role": "user", "content": "一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？" } \
  }'
```